In [35]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path
import os
import logging
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI

load_dotenv()

embeddings = OpenAIEmbeddings()
OPENAI = OpenAI()

In [17]:
BASE_DIR = Path.cwd().parent
PDF_PATH = (
    BASE_DIR
    / "Data"
    / "StudyMaterial"
    / "Complete AI"
    / "AI Agents guidebook.pdf"
)

print(PDF_PATH)

c:\Project\AI\Viru_AI_Hub\Data\StudyMaterial\Complete AI\AI Agents guidebook.pdf


In [18]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print("Pages in the document: ", len(documents))
print(documents[0].page_content[:500])

Ignoring wrong pointing object 899 0 (offset 0)


Pages in the document:  117
FREE
AI AGENTS
2025 EDITION
THE ILLUSTRATED
GUIDEBOOK 
Avi Chawla & Akshay Pachaar
DailyDoseofDS.com
Daily Dose of
Data Science


In [27]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 121


In [31]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector DB Created")

Vector DB Created


In [40]:
query = "What is guardrails and how to use them?"

In [48]:
docs = vector_db.similarity_search(
    query,
    k=1
)

In [49]:
for doc in docs:
    print(doc.page_content[:500])
    print("-" * 50)

DailyDoseofDS.com 
5) Guardrails 
Agents are powerful but without constraints, they can go oﬀ track. They might 
hallucinate, loop endlessly, or make bad calls. 
Guardrails ensure that agents stay on track and maintain quality standards. 
 
Examples of useful guardrails include: 
● Limiting tool usage: Prevent an agent from overusing APIs or generating 
irrelevant queries. 
● Setting validation checkpoints: Ensure outputs meet predeﬁned criteria 
before moving to the next step. 
● Establishing f
--------------------------------------------------


In [45]:
context = "\n\n".join([doc.page_content for doc in docs])

In [46]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

In [50]:
response = OPENAI.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Guardrails are constraints implemented to ensure that AI agents stay on track and maintain quality standards. They help prevent agents from going off track by avoiding issues such as hallucination, endless loops, or making bad calls. 

To use guardrails effectively, one can implement the following strategies:

1. **Limiting Tool Usage**: Prevent an agent from overusing APIs or generating irrelevant queries.
2. **Setting Validation Checkpoints**: Ensure outputs meet predefined criteria before allowing the agent to proceed to the next step.
3. **Establishing Fallback Mechanisms**: If an agent fails to complete a task, another agent or a human reviewer can intervene to assist. 

These guardrails are essential for maintaining the reliability and effectiveness of AI agents in real-world applications.
